In [1]:
import os 
from glob import glob
import pandas as pd

In [2]:
# 특정 경로에서 파일의 목록을 가져오는 기능 
os.listdir("./review")

['1-1.여성의류(196).json',
 '1-1.여성의류(197).json',
 '1-1.여성의류(198).json',
 '1-1.여성의류(199).json',
 '1-1.여성의류(200).json',
 '1-1.여성의류(201).json',
 '1-1.여성의류(202).json',
 '1-1.여성의류(203).json',
 '1-1.여성의류(204).json',
 '1-1.여성의류(205).json',
 '1-1.여성의류(206).json',
 '1-1.여성의류(207).json',
 '1-1.여성의류(208).json',
 '1-1.여성의류(209).json']

In [3]:
# glob 라이브러리 사용
# 장점 : 파일의 경로와 파일 명이 동시에 출력 
#       특정 확장자만 목록을 불러올수 있다. 
json_list = glob("./review/*.json")

In [4]:
# json_list를 이용하여 여러 파일들을 하나의 데이터프레임으로 결합 

# 비어있는 데이터프레임을 생성 
total_df = pd.DataFrame()

for file_path in json_list:
    # print(file_path)
    # break
    df = pd.read_json(file_path)
    # df를 total_df에 단순 행 결합 
    total_df = pd.concat( [total_df, df], axis = 0 )
total_df.reset_index(drop = True, inplace=True)
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1423 entries, 0 to 1422
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            1423 non-null   int64  
 1   RawText          1423 non-null   object 
 2   Source           1423 non-null   object 
 3   Domain           1423 non-null   object 
 4   MainCategory     1423 non-null   object 
 5   ProductName      1423 non-null   object 
 6   Syllable         1423 non-null   int64  
 7   Word             1423 non-null   int64  
 8   GeneralPolarity  1418 non-null   float64
 9   Aspects          1423 non-null   object 
dtypes: float64(1), int64(3), object(6)
memory usage: 111.3+ KB


In [5]:
total_df.head(2)

,Index,RawText,Source,Domain,MainCategory,ProductName,Syllable,Word,GeneralPolarity,Aspects
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1.0,"[{'Aspect': '디자인', 'SentimentText': '딱 기본 스타일인..."
1,1024477,드디어 겨울이 찾아왔네요. 이제부터 슬슬 겨울 패딩 장만하셔야지요? 패딩 소개해 드...,SNS,패션,여성의류,OO 아** 구스코트,464,105,1.0,"[{'Aspect': '사이즈', 'SentimentText': '저는 블랙90 사..."


In [6]:
aspect_df = pd.DataFrame(sum(total_df['Aspects'], []))

In [7]:
aspect_df.head()

,Aspect,SentimentText,SentimentWord,SentimentPolarity
0,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1
1,두께,이것만 입기엔 얇지만,3,-1
2,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1
3,색상,색상도 디자인도 무난해서,3,0
4,디자인,디자인도 무난해서,2,0


In [8]:
# 결측치가 존재하는가?
aspect_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10962 entries, 0 to 10961
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aspect             10962 non-null  object
 1   SentimentText      10962 non-null  object
 2   SentimentWord      10962 non-null  object
 3   SentimentPolarity  10962 non-null  object
dtypes: object(4)
memory usage: 342.7+ KB


In [9]:
aspect_df = aspect_df.map(lambda x : x.strip())

In [10]:
(aspect_df == '').sum()

Aspect               0
SentimentText        0
SentimentWord        0
SentimentPolarity    0
dtype: int64

In [11]:
# 종속 변수들의 데이터의 빈도수를 확인 
aspect_df['Aspect'].value_counts()

Aspect
디자인     1676
기능      1212
소재      1151
활용성      917
색상       855
핏        753
가격       679
사이즈      642
착용감      633
길이       496
품질       468
두께       353
신축성      277
무게       271
촉감       252
제품구성     169
마감       142
냄새        16
Name: count, dtype: int64

In [12]:
aspect_df['SentimentPolarity'].value_counts()

SentimentPolarity
1     9664
-1    1005
0      293
Name: count, dtype: int64

In [13]:
# 독립 변수에서 중복된 데이터가 존재하면 제거 
aspect_df.drop_duplicates('SentimentText', inplace=True)

In [14]:
aspect_df['SentimentPolarity'].value_counts()

SentimentPolarity
1     9183
-1     994
0      290
Name: count, dtype: int64

In [15]:
# 인덱스 초기화
aspect_df.reset_index(drop=True, inplace=True)

In [16]:
# 독립 변수를 토큰화 -> 벡터화 
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer

In [17]:
# 토큰화 과정에서 특정 품사들만 사용
# 명사, 동사, 형용사, 부사

okt = Okt()
allow_pos = ['Noun', 'Verb', 'Adjective', 'Adverb']

def tokenize(text):
    result = []
    for word, pos in okt.pos(text):
        if (pos in allow_pos) & (len(word) >= 2):
            result.append(word)
    return result

tokenize(
    aspect_df.loc[0, 'SentimentText']
)

['기본', '스타일', '입은', '보면', '깔끔하게', '저렴해', '보이지', '않는', '디자인']

In [18]:
vec = TfidfVectorizer(
    tokenizer= tokenize, 
    ngram_range= (1, 2), 
    min_df= 3, 
    max_df = 0.8, 
    max_features= 3000
)

In [19]:
aspect_df.columns

Index(['Aspect', 'SentimentText', 'SentimentWord', 'SentimentPolarity'], dtype='object')

In [20]:
X = aspect_df['SentimentText'].values
y1 = aspect_df['Aspect'].values
y2 = aspect_df['SentimentPolarity'].values

In [21]:
X_vec = vec.fit_transform(X)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [22]:
X_vec.shape

(10467, 3000)

In [23]:
from sklearn.preprocessing import LabelEncoder

In [24]:
le1 = LabelEncoder()
y1_le = le1.fit_transform(y1) 
le2 = LabelEncoder()
y2_le = le2.fit_transform(y2)

In [25]:
from sklearn.svm import LinearSVC

In [26]:
svc1 = LinearSVC(class_weight='balanced', random_state=42)
svc2 = LinearSVC(class_weight='balanced', random_state=42)

In [27]:
svc1.fit(X_vec, y1_le)
svc2.fit(X_vec, y2_le)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseud

In [28]:
X_vec[-500:]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2937 stored elements and shape (500, 3000)>

In [29]:
y1_le[-500:]

array([ 6, 16,  4,  7, 13,  9, 10,  5, 16,  6, 17,  9, 13, 15, 13, 10, 14,
        1,  8, 13,  7, 15,  4,  7,  1, 10, 14, 17,  0, 15, 16,  5,  0, 15,
       15,  5, 16, 11, 13,  9, 13, 15,  0,  5,  5, 10, 13,  5,  5, 15,  1,
        5, 10, 14,  9,  5,  2,  8, 13, 14, 16,  5,  5,  2,  5,  9, 14,  5,
        5,  5,  2, 10, 15, 17,  5, 11, 16,  1,  9, 10,  5,  5, 13, 17, 16,
       16, 12, 17, 10,  2,  5,  5,  0, 17, 10,  1, 14,  2,  5, 13, 17,  8,
       13,  2,  1, 13,  7,  5,  5, 17,  5,  9,  5,  5,  5,  5,  5, 15,  5,
       17,  5,  5,  9, 16, 10,  5,  0,  5,  5,  5,  5,  5, 10,  5,  5,  5,
        5,  5,  5,  5,  5,  5, 10, 13,  0,  9,  9, 16, 10,  5, 17,  9,  5,
        0, 15,  0, 17,  9,  5, 10, 10,  5,  2,  1,  5, 14,  9,  5, 10, 13,
       13,  9,  5,  1,  5, 13,  2,  0,  8,  8,  7,  1, 16, 10,  1,  2, 15,
       15,  0, 14,  9,  2,  4, 15, 16, 10, 11, 10,  1,  9,  2,  5, 10,  5,
        5, 13,  5,  5,  6, 15,  7, 17,  1,  9,  9, 14, 10,  5,  8,  5, 15,
        1,  8, 10,  1,  5

In [30]:
pred_1 = svc1.predict(X_vec[-500:])
pred_2 = svc2.predict(X_vec[-500:])

In [31]:
from sklearn.metrics import classification_report

In [32]:
print(classification_report(pred_1, y1_le[-500:]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        25
           1       0.95      0.95      0.95        40
           2       1.00      1.00      1.00        16
           3       1.00      1.00      1.00         1
           4       0.91      1.00      0.95        10
           5       0.89      0.99      0.94       110
           6       1.00      1.00      1.00         7
           7       1.00      1.00      1.00        10
           8       0.94      1.00      0.97        16
           9       0.97      0.95      0.96        40
          10       0.98      0.94      0.96        54
          11       1.00      1.00      1.00        11
          12       0.75      0.75      0.75         4
          13       1.00      0.95      0.97        39
          14       0.94      1.00      0.97        16
          15       0.93      1.00      0.96        26
          16       0.97      0.69      0.81        42
          17       1.00    

In [33]:
print(classification_report(pred_2, y2_le[-500:]))

              precision    recall  f1-score   support

           0       0.95      0.67      0.78        30
           1       0.86      0.83      0.84        23
           2       0.97      0.99      0.98       447

    accuracy                           0.97       500
   macro avg       0.93      0.83      0.87       500
weighted avg       0.97      0.97      0.96       500



- Ascept 컬럼과 SentimentPolarity 컬럼의 데이터를 결합 
- LabelEncoder를 이용하여 수치로 변환 
- Kfold, Pipe, GridsearchCV를 활용
- 계층화 폴드화는 5개 
- 벡터화, 모델 학습을 파이프라인으로 생성 
- 파라미터 조합은 벡터화에서 max_features의 조합은 (3000, None)
- SVC 모델의 조합은 C 값을 (1.0, 2.0)
- 최적의 모델의 스코어를 확인 

In [34]:
aspect_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10467 entries, 0 to 10466
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aspect             10467 non-null  object
 1   SentimentText      10467 non-null  object
 2   SentimentWord      10467 non-null  object
 3   SentimentPolarity  10467 non-null  object
dtypes: object(4)
memory usage: 327.2+ KB


In [35]:
aspect_df['Aspect'] + '_' +aspect_df['SentimentPolarity']

0        디자인_1
1        두께_-1
2         기능_1
3         색상_0
4        디자인_0
         ...  
10462    착용감_1
10463    사이즈_1
10464    디자인_1
10465    디자인_1
10466     마감_1
Length: 10467, dtype: object

In [36]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline

In [37]:
X = aspect_df['SentimentText'].values
y = (aspect_df['Aspect'] + '_' + aspect_df['SentimentPolarity']).values

In [38]:
le = LabelEncoder()
y_le = le.fit_transform(y)

In [39]:
y_le

array([17, 12,  5, ..., 17, 17, 19], shape=(10467,))

In [40]:
pipe = Pipeline(
    [
        ('vector', vec), 
        ('model', svc1)
    ]
)
cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)
params = {
    'vector__max_features' : [3000, None], 
    'model__C' : [1.0, 2.0]
}

In [41]:
grid = GridSearchCV(
    estimator= pipe, 
    param_grid= params, 
    cv = cv
)

In [42]:
grid.fit(X, y_le)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: 

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [1.0, 2.0], 'vector__max_features': [3000, None]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 :

In [43]:
grid.best_score_

np.float64(0.7412812800753501)

In [44]:
# 종속 변수를 2차원의 데이터로 그대로 활용 
X = aspect_df['SentimentText'].values
le = LabelEncoder()
aspect_df['Aspect_le'] = le.fit_transform(aspect_df['Aspect'])
aspect_df['SentimentPolarity'] = aspect_df['SentimentPolarity'].astype(int)
y = aspect_df[ ['Aspect_le', 'SentimentPolarity'] ].values

In [45]:
y.shape

(10467, 2)

In [46]:
X.shape

(10467,)

In [47]:
vec = TfidfVectorizer(
    tokenizer=tokenize, 
    ngram_range= (1,2), 
    min_df = 3, 
    max_df = 0.8, 
    lowercase= False
)

In [48]:
X_vec = vec.fit_transform(X)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [49]:
svc = LinearSVC(random_state=42, class_weight='balanced')

In [50]:
from sklearn.multioutput import MultiOutputClassifier

In [51]:
# MultiOutputClassifier는 분류모델에서 종속변수가 2차원 이상인 경우 사용하는 객체 
multi_model = MultiOutputClassifier(svc)

In [52]:
multi_model.fit(X_vec, y)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,LinearSVC(cla...ndom_state=42)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`interc

In [53]:
print(type(y[0][0]), type(y[0][1]))

<class 'numpy.int64'> <class 'numpy.int64'>


In [54]:
from sklearn.model_selection import KFold
cv2 = KFold(n_splits=5, random_state=42, shuffle=True)

In [55]:
# 쉬는시간에 확인 

pipe = Pipeline(
    [
        ('vector', vec),
        ('model', multi_model)
    ]
)
params = {
    'model__estimator__C' : [0.8, 1.0]
}
grid = GridSearchCV(
    estimator= pipe, 
    cv = cv2, 
    param_grid= params, 
    scoring= 'accuracy'
)

In [56]:
grid.fit(X, y)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=42)))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__estimator__C': [0.8, 1.0]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also di

In [57]:
pred = multi_model.predict(X_vec[-20:])

In [58]:
pred

array([[ 5,  1],
       [ 9,  1],
       [ 6,  1],
       [14,  1],
       [16,  1],
       [ 0,  1],
       [ 9,  1],
       [ 8,  1],
       [ 5, -1],
       [ 5,  1],
       [ 6, -1],
       [ 1,  1],
       [ 7,  1],
       [ 4,  1],
       [ 5,  1],
       [13,  1],
       [ 8,  1],
       [ 5,  1],
       [ 5,  1],
       [ 6,  1]])

In [59]:
pred_df = pd.DataFrame(
    pred, columns = ['aspect', 'polarity']
)
pred_df['text'] = X[-20:]

In [60]:
pred_df['aspect'] = le.inverse_transform(pred_df['aspect'])

In [61]:
pred_df

,aspect,polarity,text
0,디자인,1,디자인이 너무너무 예쁩니다.
1,색상,1,색상도 고급스러워서 마음에 들구요.
2,마감,1,바느질도 꼼꼼하게 잘 되어 있어요.
3,촉감,1,털 감촉도 아주 부드러워요~
4,핏,1,입어 보면 핏이 더 예쁜 자켓이예요~
5,가격,1,가격 대비 가성비 좋은 제품이예요~
6,색상,1,색상도 좋구요~
7,사이즈,1,모자 크기도 적당해서 마음에 들어요.
8,디자인,-1,모자가 있다 보니 밍크 특유의 올드함이 없네요.
9,디자인,1,디자인 마음에 들어요.


In [62]:
# 긴 문단을 문장 별로 나눠주기 위해 Kkma로드 
from konlpy.tag import Kkma

In [63]:
kkma = Kkma()

In [64]:
file_list = glob("./test/*.json")
file_list

['./test\\1-1.여성의류(1).json',
 './test\\1-1.여성의류(2).json',
 './test\\1-1.여성의류(3).json']

In [65]:
test_df = pd.DataFrame()

for file in file_list:
    df = pd.read_json(file)
    test_df = pd.concat( [test_df, df], axis=0 )

test_df.reset_index(drop=True, inplace=True)
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            300 non-null    int64  
 1   RawText          300 non-null    object 
 2   Source           300 non-null    object 
 3   Domain           300 non-null    object 
 4   MainCategory     300 non-null    object 
 5   ProductName      300 non-null    object 
 6   ReviewScore      300 non-null    int64  
 7   Syllable         300 non-null    int64  
 8   Word             300 non-null    int64  
 9   RDate            300 non-null    int64  
 10  GeneralPolarity  298 non-null    float64
 11  Aspects          300 non-null    object 
dtypes: float64(1), int64(5), object(6)
memory usage: 28.3+ KB


In [66]:
sample_text = test_df.loc[12, 'RawText']
sample_text

'털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고  하루입고벗었더니 겨드랑이같은곳도 엄청 뭉처있니에요 그래서 세탁을 했더니 끝도없이 손에 털뭉치???? 들이 묻어나서 포기했습니다 이게불량인가요 왜이렇게 만드신건가요 옷디자인은 너무맘에드는데 털빠짐때매 옷이아니네여'

In [67]:
# sample_text를 문장별로 나눠준다. 
text_list = kkma.sentences(sample_text)
text_list

['털이 엄청나게 빠집니다',
 '면 바바리에 엄청나게 묻어나고 하루 입고 벗었더니 겨드랑이 같은 곳도 엄청 뭉처있니에요 그래서 세탁을 했더니 끝도 없이 손에 털 뭉치???? 들이 묻어나서 포기했습니다',
 '이게 불량인가요',
 '왜 이렇게 만드신 건가요',
 '옷 디자인은 너무 맘에 드는데 털 빠짐 때 매 옷이 아니 네 여']

In [68]:
# text_list를 벡터화  -> 모델을 이용하여 예측 



In [69]:
pred = grid.predict(text_list)

In [70]:
pred

array([[16,  1],
       [15, -1],
       [15, -1],
       [ 2,  1],
       [ 5,  1]])

In [71]:
pred_df = pd.DataFrame(pred, columns = ['aspect', 'polarity'])
pred_df['text'] = text_list

pred_df

,aspect,polarity,text
0,16,1,털이 엄청나게 빠집니다
1,15,-1,면 바바리에 엄청나게 묻어나고 하루 입고 벗었더니 겨드랑이 같은 곳도 엄청 뭉처있니...
2,15,-1,이게 불량인가요
3,2,1,왜 이렇게 만드신 건가요
4,5,1,옷 디자인은 너무 맘에 드는데 털 빠짐 때 매 옷이 아니 네 여


In [72]:
pred_df['aspect'] = le.inverse_transform(pred_df['aspect'])
pred_df

,aspect,polarity,text
0,핏,1,털이 엄청나게 빠집니다
1,품질,-1,면 바바리에 엄청나게 묻어나고 하루 입고 벗었더니 겨드랑이 같은 곳도 엄청 뭉처있니...
2,품질,-1,이게 불량인가요
3,길이,1,왜 이렇게 만드신 건가요
4,디자인,1,옷 디자인은 너무 맘에 드는데 털 빠짐 때 매 옷이 아니 네 여


In [73]:
pred_df['RawText'] = sample_text
pred_df

,aspect,polarity,text,RawText
0,핏,1,털이 엄청나게 빠집니다,털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고 하루입고벗었더니 겨드랑이같은...
1,품질,-1,면 바바리에 엄청나게 묻어나고 하루 입고 벗었더니 겨드랑이 같은 곳도 엄청 뭉처있니...,털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고 하루입고벗었더니 겨드랑이같은...
2,품질,-1,이게 불량인가요,털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고 하루입고벗었더니 겨드랑이같은...
3,길이,1,왜 이렇게 만드신 건가요,털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고 하루입고벗었더니 겨드랑이같은...
4,디자인,1,옷 디자인은 너무 맘에 드는데 털 빠짐 때 매 옷이 아니 네 여,털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고 하루입고벗었더니 겨드랑이같은...


In [74]:
# test_df에서 RawText의 데이터들을 문장 별로 나눠준다. 
# raw_list 리뷰 문단을 문장으로 나눈 리스트를 담기 위한 공간 
raw_list = []
# raw_dict 리뷰 문단마다 index를 키 값으로 value는 리뷰 문단 
raw_dict = {}
for i in range(len(test_df)):
    raw_list.append(
        kkma.sentences(
            test_df.loc[i, 'RawText']
        )
    )
    raw_dict[i] = test_df.loc[i, 'RawText']

In [75]:
raw_dict

{0: '가격이 착하고 디자인이 예쁩니다',
 1: '싸고  디자인이 예뻐요. . 정말  가성비 가심비 입니다',
 2: '편하고  디자인이 예뻐요  가격도  좋아요   시원해요  빨리 마르고  이것만  입게되요',
 3: '너무 착한가격에 감사합니다 윈하는 색은 없지만',
 4: '가격이  너무 좋아서  블랙 구매했습니다  그런데 소재도  맘에  들어  흰색도  구매했습니다',
 5: '외출할때 입기에는 재질도 디자인도 좀~그러네요 그냥 집에서 편하게 입을수는 있을것 같아요',
 6: '특히 자켓이 넘작고 짧으네요. 상의는 좀작구 바지는 잘맞아요.',
 7: '싸고품질이좋아요ᆢ사이즈는좀큰듯하구요ᆢ반사이즈는내려서주문하심좋을듯해요',
 8: '바지가 너무 편하고 좋아요 티셔츠는 여름에 잘 입을 듯 베스트는 디자인은 별로지만 가격이 착해서 만족',
 9: '디자인이 예뻐요. 사이즈 잘 맞습니다.',
 10: '사이즈 딱 맞구요.  너무 예쁘고  얇지도 두껍지도 않으네요.',
 11: '저렴하게 구입해서 겨울에 잘 입을거 같아요',
 12: '털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고  하루입고벗었더니 겨드랑이같은곳도 엄청 뭉처있니에요 그래서 세탁을 했더니 끝도없이 손에 털뭉치???? 들이 묻어나서 포기했습니다 이게불량인가요 왜이렇게 만드신건가요 옷디자인은 너무맘에드는데 털빠짐때매 옷이아니네여',
 13: '디자인이 여성스러운 분위기가 납니다 구겨져와서 다림질 필요했어요',
 14: '99사이즈 입는데 88 사이즈 잘맞고~ 기장이 살짝 짧은듯 하지만 밝은색도 있고~ 까실까실하니 좋으네요~',
 15: '4종 마음에 들어요 색깔도 이쁘고 여름니트 입니다',
 16: '색상이 예쁘고  구김이없어요',
 17: '색상 디자인 예쁘고 맘에들어요',
 18: '가성비 갑입니다 가로패턴이라 더커 보이면 어쩌나 싶었는데 칼라고 디자인이고 다 만족합니다  가슴도  좀있는편이라  걱정했는데 핏이 괸찮아요  비치지 않아서 하나만 입고 다녀도될만 합니다',
 19: '오 좋

In [76]:
# raw_list
sentence_df = pd.DataFrame()
for idx, raw in enumerate(raw_list):
    pred = grid.predict(raw)
    temp_df = pd.DataFrame(pred, columns = ['Pred_Aspect', 'Pred_Polarity'])
    # raw_dict의 idx의 값은 해당 예측 값의 문단
    temp_df['sentence_text'] = raw
    temp_df['RawText'] = raw_dict[idx]
    # temp_df는 하나의 리뷰에 대한 예측 값들
    # print(temp_df)
    # break
    # 나온 데이터프레임을 sentence_df에 단순 행 결합 
    sentence_df = pd.concat( [sentence_df, temp_df], axis=0 )
sentence_df.reset_index(drop=True, inplace=True)

In [77]:
sentence_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 707 entries, 0 to 706
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Pred_Aspect    707 non-null    int64 
 1   Pred_Polarity  707 non-null    int64 
 2   sentence_text  707 non-null    object
 3   RawText        707 non-null    object
dtypes: int64(2), object(2)
memory usage: 22.2+ KB


In [78]:
sentence_df.head(10)

,Pred_Aspect,Pred_Polarity,sentence_text,RawText
0,5,1,가격이 착하고 디자인이 예쁩니다,가격이 착하고 디자인이 예쁩니다
1,5,1,싸고 디자인이 예뻐요. .,싸고 디자인이 예뻐요. . 정말 가성비 가심비 입니다
2,16,1,정말 가 성비 가심 비 입니다,싸고 디자인이 예뻐요. . 정말 가성비 가심비 입니다
3,5,1,편하고 디자인이 예뻐요,편하고 디자인이 예뻐요 가격도 좋아요 시원해요 빨리 마르고 이것만 입게되요
4,0,1,가격도 좋아요,편하고 디자인이 예뻐요 가격도 좋아요 시원해요 빨리 마르고 이것만 입게되요
5,15,1,시 원해요 빨리 마르고 이것만 입게 되요,편하고 디자인이 예뻐요 가격도 좋아요 시원해요 빨리 마르고 이것만 입게되요
6,0,1,너무 착한 가격에 감사합니다,너무 착한가격에 감사합니다 윈하는 색은 없지만
7,5,-1,윈하는 색은 없지만,너무 착한가격에 감사합니다 윈하는 색은 없지만
8,0,1,가격이 너무 좋아서 블랙 구매했습니다,가격이 너무 좋아서 블랙 구매했습니다 그런데 소재도 맘에 들어 흰색도 ...
9,10,-1,그런데 소재도 맘에 들어 흰색도 구매했습니다,가격이 너무 좋아서 블랙 구매했습니다 그런데 소재도 맘에 들어 흰색도 ...


In [79]:
sentence_df['Pred_Aspect'] = le.inverse_transform(sentence_df['Pred_Aspect'])

In [80]:
sentence_df.head(10)

,Pred_Aspect,Pred_Polarity,sentence_text,RawText
0,디자인,1,가격이 착하고 디자인이 예쁩니다,가격이 착하고 디자인이 예쁩니다
1,디자인,1,싸고 디자인이 예뻐요. .,싸고 디자인이 예뻐요. . 정말 가성비 가심비 입니다
2,핏,1,정말 가 성비 가심 비 입니다,싸고 디자인이 예뻐요. . 정말 가성비 가심비 입니다
3,디자인,1,편하고 디자인이 예뻐요,편하고 디자인이 예뻐요 가격도 좋아요 시원해요 빨리 마르고 이것만 입게되요
4,가격,1,가격도 좋아요,편하고 디자인이 예뻐요 가격도 좋아요 시원해요 빨리 마르고 이것만 입게되요
5,품질,1,시 원해요 빨리 마르고 이것만 입게 되요,편하고 디자인이 예뻐요 가격도 좋아요 시원해요 빨리 마르고 이것만 입게되요
6,가격,1,너무 착한 가격에 감사합니다,너무 착한가격에 감사합니다 윈하는 색은 없지만
7,디자인,-1,윈하는 색은 없지만,너무 착한가격에 감사합니다 윈하는 색은 없지만
8,가격,1,가격이 너무 좋아서 블랙 구매했습니다,가격이 너무 좋아서 블랙 구매했습니다 그런데 소재도 맘에 들어 흰색도 ...
9,소재,-1,그런데 소재도 맘에 들어 흰색도 구매했습니다,가격이 너무 좋아서 블랙 구매했습니다 그런데 소재도 맘에 들어 흰색도 ...


In [81]:
# test_df와 sentence_df을 조인 결합 
# (test_df와 sentence_df의 RawText는 같은 집합 -> 4개의 종류는 모두 같은 결과가 나온다.)
len(
    pd.merge( test_df, sentence_df, on = 'RawText', how='outer' )
)

707

In [82]:
df = pd.merge(test_df, sentence_df, on = 'RawText', how='inner')

In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 707 entries, 0 to 706
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            707 non-null    int64  
 1   RawText          707 non-null    object 
 2   Source           707 non-null    object 
 3   Domain           707 non-null    object 
 4   MainCategory     707 non-null    object 
 5   ProductName      707 non-null    object 
 6   ReviewScore      707 non-null    int64  
 7   Syllable         707 non-null    int64  
 8   Word             707 non-null    int64  
 9   RDate            707 non-null    int64  
 10  GeneralPolarity  701 non-null    float64
 11  Aspects          707 non-null    object 
 12  Pred_Aspect      707 non-null    object 
 13  Pred_Polarity    707 non-null    int64  
 14  sentence_text    707 non-null    object 
dtypes: float64(1), int64(6), object(8)
memory usage: 83.0+ KB


In [84]:
df.head(10)

,Index,RawText,Source,Domain,MainCategory,ProductName,ReviewScore,Syllable,Word,RDate,GeneralPolarity,Aspects,Pred_Aspect,Pred_Polarity,sentence_text
0,7,가격이 착하고 디자인이 예쁩니다,쇼핑몰,패션,여성의류,OO 플** 베스트 풀코디 3종,100,17,4,20210815,1.0,"[{'Aspect': '가격', 'SentimentText': '가격이 착하고', ...",디자인,1,가격이 착하고 디자인이 예쁩니다
1,9,싸고 디자인이 예뻐요. . 정말 가성비 가심비 입니다,쇼핑몰,패션,여성의류,OO 플** 베스트 풀코디 3종,100,31,8,20210807,1.0,"[{'Aspect': '가격', 'SentimentText': '싸고', 'Sent...",디자인,1,싸고 디자인이 예뻐요. .
2,9,싸고 디자인이 예뻐요. . 정말 가성비 가심비 입니다,쇼핑몰,패션,여성의류,OO 플** 베스트 풀코디 3종,100,31,8,20210807,1.0,"[{'Aspect': '가격', 'SentimentText': '싸고', 'Sent...",핏,1,정말 가 성비 가심 비 입니다
3,15,편하고 디자인이 예뻐요 가격도 좋아요 시원해요 빨리 마르고 이것만 입게되요,쇼핑몰,패션,여성의류,OO 플** 베스트 풀코디 3종,100,49,10,20210804,1.0,"[{'Aspect': '착용감', 'SentimentText': '편하고', 'Se...",디자인,1,편하고 디자인이 예뻐요
4,15,편하고 디자인이 예뻐요 가격도 좋아요 시원해요 빨리 마르고 이것만 입게되요,쇼핑몰,패션,여성의류,OO 플** 베스트 풀코디 3종,100,49,10,20210804,1.0,"[{'Aspect': '착용감', 'SentimentText': '편하고', 'Se...",가격,1,가격도 좋아요
5,15,편하고 디자인이 예뻐요 가격도 좋아요 시원해요 빨리 마르고 이것만 입게되요,쇼핑몰,패션,여성의류,OO 플** 베스트 풀코디 3종,100,49,10,20210804,1.0,"[{'Aspect': '착용감', 'SentimentText': '편하고', 'Se...",품질,1,시 원해요 빨리 마르고 이것만 입게 되요
6,17,너무 착한가격에 감사합니다 윈하는 색은 없지만,쇼핑몰,패션,여성의류,OO 플** 베스트 풀코디 3종,100,25,6,20210803,1.0,"[{'Aspect': '가격', 'SentimentText': '너무 착한가격에 감...",가격,1,너무 착한 가격에 감사합니다
7,17,너무 착한가격에 감사합니다 윈하는 색은 없지만,쇼핑몰,패션,여성의류,OO 플** 베스트 풀코디 3종,100,25,6,20210803,1.0,"[{'Aspect': '가격', 'SentimentText': '너무 착한가격에 감...",디자인,-1,윈하는 색은 없지만
8,18,가격이 너무 좋아서 블랙 구매했습니다 그런데 소재도 맘에 들어 흰색도 ...,쇼핑몰,패션,여성의류,OO 플** 베스트 풀코디 3종,100,52,11,20210717,1.0,"[{'Aspect': '가격', 'SentimentText': '가격이 너무 좋아...",가격,1,가격이 너무 좋아서 블랙 구매했습니다
9,18,가격이 너무 좋아서 블랙 구매했습니다 그런데 소재도 맘에 들어 흰색도 ...,쇼핑몰,패션,여성의류,OO 플** 베스트 풀코디 3종,100,52,11,20210717,1.0,"[{'Aspect': '가격', 'SentimentText': '가격이 너무 좋아...",소재,-1,그런데 소재도 맘에 들어 흰색도 구매했습니다


In [85]:
# ProductName의 고유 값들을 확인 
df['ProductName'].unique()

array(['OO 플** 베스트 풀코디 3종', 'OO 여성 소프트 진주 아** 니트 2종 택1',
       'OO 크** 긴팔 니트 티셔츠 베이지', 'OO 샤** 썸머 쿨니트 4종', 'OO 레** 카라 상하세트 3종 택1',
       'OO 여성 인견 원피스 3종세트', 'OO 여성 시** 실크코튼 티셔츠 4종 세트',
       'OO 클** 카라 반팔티셔츠 12종 택1', 'OO 울블랜디드 여성 니트 3종', 'OO 라** 니트 4종세트',
       'OO 웜 소프트 부** 니트 3종세트', 'OO 여성 밍** 캐시미어 니트 4종 택1', 'OO 여성 니트 3종',
       'OO 니트 라** 세트', 'OO 스** 티셔츠 4종세트', 'OO 울블랜디드 니트 4종 세트',
       'OO 스** 케이블 니트 풀오버 4종세트', 'OO 레이어드 풀오버 4종 세트', 'OO 여성 골프니트3종',
       'OO 기모 니트 4종세트', 'OO 캐시미어코튼 글** 니트 3종 택1', 'OO 니트숄',
       'OO 기모 풀오버 3종 SET ', 'OO 남녀 겨** 니트 골프웨어 골프복 4종세트', 'OO 모** 니트 2종',
       'OO 여성 호** 혼방 풀오버 1+1 2종세트', 'OO 여성 기모 맨투맨티셔츠 1+1 2종세트',
       'OO 봄 케이블니트 니트 3종', 'OO 봄 여성 케이블니트 니트 3종', 'OO 스프링 니트블라우스 3종',
       'OO 스프링 니트 블라우스 3종_리뉴얼', 'OO 여성 패딩자켓+티셔츠3종',
       '블** 겨울 OO 마스터 컬렉션 여성 4종(니트/스웨터)', 'OO 스** 니트블라우스 4종',
       '여성 OO 시** 가을 코튼니트 3종', 'OO 캐시미어 블** 숄가디건',
       'OO 여성 하** 웜웨어 스웨터 4종세트', 'OO 여성 자켓+니트티셔츠 3종',
       'OO 여성 니트자켓+니트티셔츠 총 4종', 'OO 니트자켓+니트티셔츠 

In [86]:
product_df = df.loc[
    df['ProductName'] == 'OO 플** 베스트 풀코디 3종'
]

In [87]:
# Pred_Aspect 컬럼으로 그룹화를 하고 Pred_Polarity 의 평균 
product_df[['Pred_Aspect', 'Pred_Polarity']].groupby('Pred_Aspect').mean()

,Pred_Polarity
Pred_Aspect,
가격,1.000000
디자인,0.200000
사이즈,-0.333333
소재,-1.000000
착용감,1.000000
품질,1.000000
핏,1.000000


In [93]:
# df에서 그룹화를 ProdictName, Pred_Aspect기준으로 그룹화를 하고 Pred_Polarity 의 평균 
df[ ['ProductName', 'Pred_Aspect', 'Pred_Polarity'] ]\
    .groupby(['ProductName', 'Pred_Aspect']).mean().unstack()

Pred_Polarity                                \
Pred_Aspect                                가격        기능        길이   냄새   두께   
ProductName                                                                   
OO 기모 니트 4종세트                        0.666667  1.000000       NaN  NaN  1.0   
OO 기모 풀오버 3종 SET                     1.000000       NaN  1.000000 -1.0  1.0   
OO 남녀 겨** 니트 골프웨어 골프복 4종세트                NaN       NaN       NaN  NaN  NaN   
OO 니트 라** 세트                         1.000000       NaN -1.000000  NaN  NaN   
OO 니트숄                               1.000000       NaN       NaN  NaN  NaN   
OO 니트자켓+니트티셔츠 총 4종                        NaN       NaN       NaN  NaN  NaN   
OO 라** 니트 4종세트                            NaN       NaN       NaN  NaN  NaN   
OO 레** 카라 상하세트 3종 택1                 1.000000  1.000000       NaN  NaN  1.0   
OO 레이어드 풀오버 4종 세트                    0.600000       NaN  1.000000  NaN  1.0   
OO 루** 니트 셋업 세트                      1.000000       NaN       NaN  NaN  NaN   
OO 모** 니트 2종                              NaN       NaN       NaN  NaN  NaN   
OO 봄 여성 케이블니트 니트 3종                       NaN  1.000000  1.000000  NaN  0.5   
OO 봄 케이블니트 니트 3종                     1.000000       NaN       NaN  NaN  NaN   
OO 샤** 썸머 쿨니트 4종                     1.000000       NaN       NaN  NaN  NaN   
OO 스** 니트블라우스 4종                     1.000000       NaN  1.000000  NaN  1.0   
OO 스** 케이블 니트 풀오버 4종세트               1.000000       NaN       NaN  NaN  NaN   
OO 스** 티셔츠 4종세트                      1.000000  0.333333  1.000000  NaN  NaN   
OO 스프링 니트 블라우스 3종_리뉴얼                     NaN       NaN       NaN  NaN  NaN   
OO 스프링 니트블라우스 3종                     1.000000  1.000000       NaN  NaN  NaN   
OO 여성 골프니트3종                         0.333333  1.000000 -0.666667  NaN  NaN   
OO 여성 기모 맨투맨티셔츠 1+1 2종세트                  NaN       NaN       NaN  NaN  NaN   
OO 여성 니트 3종                               NaN  1.000000       NaN  NaN  0.0   
OO 여성 니트자켓+니트티셔츠 총 4종                0.400000  1.000000  0.000000  NaN -1.0   
OO 여성 밍** 캐시미어 니트 4종 택1              1.000000  1.000000  0.000000  NaN  NaN   
OO 여성 소프트 진주 아** 니트 2종 택1            1.000000       NaN  1.000000  NaN  1.0   
OO 여성 시** 실크코튼 티셔츠 4종 세트                  NaN  1.000000       NaN  NaN  1.0   
OO 여성 인견 원피스 3종세트                    1.000000  1.000000  1.000000  NaN  NaN   
OO 여성 자켓+니트티셔츠 3종                         NaN  1.000000       NaN  NaN  NaN   
OO 여성 패딩자켓+티셔츠3종                          NaN       NaN -1.000000  NaN  NaN   
OO 여성 하** 웜웨어 스웨터 4종세트                    NaN       NaN       NaN  NaN  NaN   
OO 여성 호** 혼방 풀오버 1+1 2종세트            1.000000       NaN       NaN  NaN  NaN   
OO 울 블** 롱베스트+ 니트 풀오버 세트             1.000000       NaN  0.000000  NaN  1.0   
OO 울블랜디드 니트 4종 세트                    0.428571  1.000000 -1.000000 -1.0  1.0   
OO 울블랜디드 여성 니트 3종                    1.000000       NaN       NaN  NaN  0.5   
OO 웜 소프트 부** 니트 3종세트                 0.875000  1.000000 -1.000000  NaN  NaN   
OO 캐시미어 블** 숄가디건                     1.000000  1.000000  0.000000  NaN  1.0   
OO 캐시미어코튼 글** 니트 3종 택1               1.000000       NaN  0.500000  NaN  1.0   
OO 크** 긴팔 니트 티셔츠 베이지                      NaN       NaN       NaN  NaN  NaN   
OO 클** 카라 반팔티셔츠 12종 택1               1.000000       NaN       NaN  NaN  NaN   
OO 플** 베스트 풀코디 3종                    1.000000       NaN       NaN  NaN  NaN   
블** 겨울 OO 마스터 컬렉션 여성 4종(니트/스웨터)           NaN       NaN       NaN  NaN  NaN   
여성 OO 시** 가을 코튼니트 3종                 0.666667  0.500000  1.000000  NaN  NaN   

                                                                         \
Pred_Aspect                           디자인   마감   무게       사이즈        색상   
ProductName                                                               
OO 기모 니트 4종세트                    1.000000  NaN  1.0 -1.000000  1.000000   
OO 기모 풀오버 3종 SET                 1.000000  NaN  NaN -1.000000 -1.000000   
OO 남녀 겨** 니트 골프웨어 골프복 4종세트            NaN  NaN  NaN       NaN  1.000000   
OO 니트 라** 세트              

In [92]:
pd.pivot_table(
    df, 
    index = 'ProductName',
    columns = 'Pred_Aspect' ,
    values = 'Pred_Polarity', 
    aggfunc= 'mean'
)

Pred_Aspect,가격,기능,길이,냄새,두께,디자인,마감,무게,사이즈,색상,소재,신축성,제품구성,착용감,촉감,품질,핏,활용성
ProductName,,,,,,,,,,,,,,,,,,
OO 기모 니트 4종세트,0.666667,1.000000,NaN,NaN,1.0,1.000000,NaN,1.0,-1.000000,1.000000,-0.500000,NaN,NaN,1.000000,NaN,1.000000,1.000000,0.000000
OO 기모 풀오버 3종 SET,1.000000,NaN,1.000000,-1.0,1.0,1.000000,NaN,NaN,-1.000000,-1.000000,NaN,NaN,NaN,1.000000,NaN,1.000000,1.000000,1.000000
OO 남녀 겨** 니트 골프웨어 골프복 4종세트,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN
OO 니트 라** 세트,1.000000,NaN,-1.000000,NaN,NaN,1.000000,NaN,NaN,0.000000,1.000000,0.000000,NaN,NaN,1.000000,1.000000,1.000000,NaN,1.000000
OO 니트숄,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,1.000000,1.000000,NaN,NaN,NaN
OO 니트자켓+니트티셔츠 총 4종,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,1.0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OO 라** 니트 4종세트,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OO 레** 카라 상하세트 3종 택1,1.000000,1.000000,NaN,NaN,1.0,1.000000,NaN,NaN,-0.500000,-1.000000,-1.000000,NaN,NaN,1.000000,1.000000,1.000000,0.000000,NaN
OO 레이어드 풀오버 4종 세트,0.600000,NaN,1.000000,NaN,1.0,1.000000,NaN,NaN,1.000000,1.000000,NaN,-1.000000,1.0,NaN,1.000000,0.000000,0.500000,1.000000
